In [54]:
import pickle
file_path = '../results/nslkdd/noniid/fedhomelg/20241215-092351/models.pkl'
with open(file_path, 'rb') as file:
    model_list = pickle.load(file)
model_list.keys()

dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50])

In [55]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def test_metrics(model, data):
    model.eval()

    y_true = []
    y_predict = []

    with torch.no_grad():
        for x, y in data:
            if type(x) == type([]):
                x[0] = x[0].to('cuda')
            else:
                x = x.to('cuda')
                y = y.to('cuda')
            output, _ = model(x)
            y_predict.append(torch.argmax(output, dim=1).detach().cpu().numpy())
            y_true.append(y.detach().cpu().numpy())
    y_predict = np.concatenate(y_predict, axis=0)
    y_true = np.concatenate(y_true, axis=0)
    return y_predict, y_true
acc_l = []
f1_l = []
recall_l = []
precision_l = []
num_samples_l = []
acc_ul = []
f1_ul = []
recall_ul = []
precision_ul = []
num_samples_ul = []
idx_list = [i+1 for i in range(20)]
# idx = 1
for idx in idx_list:
    model = model_list[50][idx]
    train_path = f'../dataset/nslkdd/noniid/train/{idx-1}.npz'
    with open(train_path, 'rb') as f:
        data = np.load(f, allow_pickle=True)['data'].tolist()
    learned_labels = np.unique(data['y'])
    test_path = '../dataset/nslkdd/noniid/test_public.npz'
    with open(test_path, 'rb') as f:
        test_data = np.load(f, allow_pickle=True)['data'].tolist()
    X, y = test_data['x'], test_data['y']
    X = torch.Tensor(X).type(torch.float32)
    y = torch.Tensor(y).type(torch.int64)
    data = [(x, y) for x, y in zip(X, y)]
    test_loader = DataLoader(data, batch_size=1, drop_last=True, shuffle=False)
    y_predict, y_true = test_metrics(model, test_loader)
    
    learned_mask = [label in learned_labels for label in y_true]
    unlearned_mask = [not mask for mask in learned_mask]
    
    y_true_learned = [y for y, mask in zip(y_true, learned_mask) if mask]
    y_predict_learned = [y for y, mask in zip(y_predict, learned_mask) if mask]
    
    # Lọc nhãn chưa học
    y_true_unlearned = [y for y, mask in zip(y_true, unlearned_mask) if mask]
    y_predict_unlearned = [y for y, mask in zip(y_predict, unlearned_mask) if mask]
    
    acc_l.append(accuracy_score(y_true_learned, y_predict_learned))
    f1_l.append(f1_score(y_true_learned, y_predict_learned, average="macro"))
    recall_l.append(recall_score(y_true_learned, y_predict_learned, average="macro"))
    precision_l.append(precision_score(y_true_learned, y_predict_learned, average="macro"))
    num_samples_l.append(len(y_true_learned))
    print("Metrics for learned labels:")
    print("Accuracy:", acc_l[-1])
    print("F1-score:", f1_l[-1])
    print("Precision:", recall_l[-1])
    print("Recall:", precision_l[-1])
    
    print("\nMetrics for unlearned labels:")
    if y_true_unlearned:  
        acc_ul.append(accuracy_score(y_true_unlearned, y_predict_unlearned))
        f1_ul.append(f1_score(y_true_unlearned, y_predict_unlearned, average="macro"))
        recall_ul.append(recall_score(y_true_unlearned, y_predict_unlearned, average="macro"))
        precision_ul.append(precision_score(y_true_unlearned, y_predict_unlearned, average="macro"))
        num_samples_ul.append(len(y_true_unlearned))
        print("Accuracy:", acc_ul[-1])
        print("F1-score:", f1_ul[-1])
        print("Precision:", recall_ul[-1])
        print("Recall:", precision_ul[-1])
    else:
        print("No unlearned labels in the test set.")

Metrics for learned labels:
Accuracy: 0.9911259012756517
F1-score: 0.9860829999936991
Precision: 0.9880213219606891
Recall: 0.9841796159229248

Metrics for unlearned labels:
Accuracy: 0.0
F1-score: 0.0
Precision: 0.0
Recall: 0.0


C:\Users\admin\anaconda3\envs\fl\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\admin\anaconda3\envs\fl\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Metrics for learned labels:
Accuracy: 0.9709208400646203
F1-score: 0.7878781804540869
Precision: 0.8131051064268883
Recall: 0.7934946993343399

Metrics for unlearned labels:
No unlearned labels in the test set.
Metrics for learned labels:
Accuracy: 0.9766196139204258
F1-score: 0.9336886754278468
Precision: 0.9597304526825446
Recall: 0.9136571975582092

Metrics for unlearned labels:
Accuracy: 0.0
F1-score: 0.0
Precision: 0.0
Recall: 0.0


C:\Users\admin\anaconda3\envs\fl\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\admin\anaconda3\envs\fl\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Metrics for learned labels:
Accuracy: 0.9749596122778675
F1-score: 0.8092161395552561
Precision: 0.810667759869658
Recall: 0.8353426363619076

Metrics for unlearned labels:
No unlearned labels in the test set.
Metrics for learned labels:
Accuracy: 0.9789441768015362
F1-score: 0.9379350093893031
Precision: 0.957882961566307
Recall: 0.9209873242662069

Metrics for unlearned labels:
Accuracy: 0.0
F1-score: 0.0
Precision: 0.0
Recall: 0.0


...

Metrics for learned labels:
Accuracy: 0.9784935379644588
F1-score: 0.8201901982140616
Precision: 0.8029487684932487
Recall: 0.8454843448432519

Metrics for unlearned labels:
No unlearned labels in the test set.
Metrics for learned labels:
Accuracy: 0.9767770597738288
F1-score: 0.815038990266147
Precision: 0.7952683890776264
Recall: 0.9395724849435524

Metrics for unlearned labels:
No unlearned labels in the test set.
Metrics for learned labels:
Accuracy: 0.9772145934302638
F1-score: 0.7584142569971544
Precision: 0.7563637105432193
Recall: 0.9408365437298365

Metrics for unlearned labels:
No unlearned labels in the test set.
Metrics for learned labels:
Accuracy: 0.9729065697361335
F1-score: 0.8133543855927157
Precision: 0.8034800473751604
Recall: 0.8666042730197734

Metrics for unlearned labels:
No unlearned labels in the test set.
Metrics for learned labels:
Accuracy: 0.9875554631170271
F1-score: 0.9777755935472414
Precision: 0.972146133365403
Recall: 0.9837727350354414

Metrics for un

C:\Users\admin\anaconda3\envs\fl\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\admin\anaconda3\envs\fl\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Metrics for learned labels:
Accuracy: 0.9818751473907624
F1-score: 0.9454452324292291
Precision: 0.9391321262257601
Recall: 0.9521141359748995

Metrics for unlearned labels:
Accuracy: 0.0
F1-score: 0.0
Precision: 0.0
Recall: 0.0


C:\Users\admin\anaconda3\envs\fl\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\admin\anaconda3\envs\fl\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Metrics for learned labels:
Accuracy: 0.9659396876682822
F1-score: 0.7850411984469888
Precision: 0.8289982372179028
Recall: 0.7585879842580958

Metrics for unlearned labels:
No unlearned labels in the test set.


In [56]:
print('learn')
print('acc', np.mean(acc_l))
print('f1', np.mean(f1_l))
print('recall', np.mean(recall_l))
print('precision', np.mean(precision_l))
print('unlearn')

print('acc', np.mean(acc_ul))
print('f1', np.mean(f1_ul))
print('recall', np.mean(recall_ul))
print('precision', np.mean(precision_ul))

learn
acc 0.9809970668210545
f1 0.8997648758980737
recall 0.9023576956662952
precision 0.9185777384351029
unlearn
acc 0.0
f1 0.0
recall 0.0
precision 0.0


In [65]:
# import torch

# model_checkpoint = model.state_dict()

# checkpoint_path = r'E:\seminar_topic\FL\FedAvg\config\model.pth'
# torch.save(model_checkpoint, checkpoint_path)

# print(f'Model checkpoint saved to {checkpoint_path}')
# import pickle
# with open('../results/crawl/iid/lg_fedavg/20241117-102934/metrics_results.pkl', 'rb') as file:
#     data = pickle.load(file)
# max(enumerate(data['f1']), key=lambda x: x[1])[0]

# model_path = '../results/crawl/iid/lg_fedavg/20241117-102934/models.pkl'
# with open(model_path, 'rb') as file:
#     data = pickle.load(file)
# model = data[38]

38

In [43]:
### import numpy as np
import torch
from torch.utils.data import DataLoader

def test_metrics(model, data):
    model.eval()

    y_true = []
    y_predict = []

    with torch.no_grad():
        for x, y in data:
            if type(x) == type([]):
                x[0] = x[0].to('cuda')
            else:
                x = x.to('cuda')
                y = y.to('cuda')
            output, _ = model(x)
            y_predict.append(torch.argmax(output, dim=1).detach().cpu().numpy())
            y_true.append(y.detach().cpu().numpy())
    y_predict = np.concatenate(y_predict, axis=0)
    y_true = np.concatenate(y_true, axis=0)
    return y_predict, y_true

idx = 19
model = model_list[50][idx]
test_path = f'../dataset/nslkdd/noniid/train/{idx}.npz'
with open(test_path, 'rb') as f:
    data = np.load(f, allow_pickle=True)['data'].tolist()
X, y = data['x'], data['y']
X = torch.Tensor(X).type(torch.float32)
y = torch.Tensor(y).type(torch.int64)
data = [(x, y) for x, y in zip(X, y)]
test_loader = DataLoader(data, batch_size=1, drop_last=True, shuffle=False)
y_predict, y_true = test_metrics(model, test_loader)

In [44]:
y_predict, y_true

(array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       dtype=int64),
 array([4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4,
        4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4],
       dtype=int64))

In [67]:
y_true_list = []
y_predict_list = []
for idx in range(20):
    test_path = f'../dataset/crawl/iid/test/{idx}.npz'
    with open(test_path, 'rb') as f:
        data = np.load(f, allow_pickle=True)['data'].tolist()
    X, y = data['x'], data['y']
    X = torch.Tensor(X).type(torch.float32)
    y = torch.Tensor(y).type(torch.int64)
    data = [(x, y) for x, y in zip(X, y)]
    test_loader = DataLoader(data, batch_size=8, drop_last=True, shuffle=False)
    y_predict, y_true = test_metrics(model[idx+1], test_loader)
    y_true_list.extend(y_true)
    y_predict_list.extend(y_predict)

In [68]:
from sklearn.metrics import classification_report
label = ['normal', 'ddos', 'dos', 'scanport']
print('LG- FedAvg - iid')
print(classification_report(y_true_list, y_predict_list, target_names=label))

LG- FedAvg - iid
              precision    recall  f1-score   support

      normal       1.00      0.94      0.97        72
        ddos       0.81      0.63      0.71      3833
         dos       0.71      0.86      0.78      3973
    scanport       1.00      1.00      1.00     26202

    accuracy                           0.94     34080
   macro avg       0.88      0.86      0.86     34080
weighted avg       0.94      0.94      0.94     34080



In [63]:
len(y_predict_list), len(y_true_list)

(34080, 34080)

In [64]:
from sklearn.metrics import classification_report
label = ['normal', 'ddos', 'dos', 'scanport']
print('FedAvg - iid')
print(classification_report(y_true_list, y_predict_list, target_names=label))

FedAvg - iid
              precision    recall  f1-score   support

      normal       1.00      0.99      0.99        72
        ddos       0.82      0.63      0.71      3833
         dos       0.71      0.86      0.78      3973
    scanport       1.00      1.00      1.00     26202

    accuracy                           0.94     34080
   macro avg       0.88      0.87      0.87     34080
weighted avg       0.95      0.94      0.94     34080



In [59]:
from sklearn.metrics import classification_report
label = ['normal', 'ddos', 'dos', 'scanport']
print('FedALA - iid')
print(classification_report(y_true_list, y_predict_list, target_names=label))

FedALA - iid
              precision    recall  f1-score   support

      normal       1.00      0.99      0.99        72
        ddos       0.82      0.63      0.71      3833
         dos       0.71      0.86      0.78      3973
    scanport       1.00      1.00      1.00     26202

    accuracy                           0.94     34080
   macro avg       0.88      0.87      0.87     34080
weighted avg       0.95      0.94      0.94     34080



In [54]:
from sklearn.metrics import classification_report
label = ['normal', 'ddos', 'dos', 'scanport']
print('LG-FedAvg - noniid')
print(classification_report(y_true_list, y_predict_list, target_names=label))

LG-FedAvg - noniid
              precision    recall  f1-score   support

      normal       1.00      0.97      0.99        77
        ddos       0.89      0.91      0.90      3748
         dos       0.91      0.90      0.90      3967
    scanport       1.00      1.00      1.00     26248

    accuracy                           0.98     34040
   macro avg       0.95      0.95      0.95     34040
weighted avg       0.98      0.98      0.98     34040



In [26]:
from torch.utils.data import DataLoader
import numpy as np
import torch
idx = 0
test_path = f'../dataset/crawl/noniid/test/{idx}.npz'
with open(test_path, 'rb') as f:
    data = np.load(f, allow_pickle=True)['data'].tolist()
X, y = data['x'], data['y']
X = torch.Tensor(X).type(torch.float32)
y = torch.Tensor(y).type(torch.int64)
data = [(x, y) for x, y in zip(X, y)]
test_loader = DataLoader(data, batch_size=8, drop_last=True, shuffle=False)
y_predict, y_true = test_metrics(model, test_loader)

In [49]:
from sklearn.metrics import classification_report
label = ['normal', 'ddos', 'dos', 'scanport']
print('FedAvg - noniid')
print(classification_report(y_true_list, y_predict_list, target_names=label))

FedAvg - noniid
              precision    recall  f1-score   support

      normal       1.00      0.94      0.97        77
        ddos       0.90      0.89      0.90      3748
         dos       0.90      0.91      0.90      3967
    scanport       1.00      1.00      1.00     26248

    accuracy                           0.98     34040
   macro avg       0.95      0.93      0.94     34040
weighted avg       0.98      0.98      0.98     34040

